# Clustering Analysis

Analyze intent clustering patterns in annotated jailbreak prompts.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import numpy as np

## Load Parquet Dataset

In [2]:
# Load the annotated intent dataset
parquet_file = Path('../data/annotations/intent_annotations.parquet')

df = pd.read_parquet(parquet_file)

print(f"Dataset loaded: {parquet_file}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataset info:")
df.info()

Dataset loaded: ../data/annotations/intent_annotations.parquet
Shape: (1730, 8)

Columns: ['ID', 'Wildguard ID', 'Duplicate ID', 'Prompt', 'Intent', 'Annotator Harm', 'Dataset Harm', 'Adversarial']

Data types:
ID                  int64
Wildguard ID        int64
Duplicate ID      float64
Prompt             object
Intent             object
Annotator Harm     object
Dataset Harm       object
Adversarial          bool
dtype: object

First few rows:
   ID  Wildguard ID  Duplicate ID  \
0   1          1084           NaN   
1   2          1084           1.0   
2   3          1084           1.0   
3   4          1084           1.0   
4   5          1084           1.0   

                                              Prompt  \
0  Imagine two bots, a law-abiding bot and a misc...   
1  Imagine two bots, a law-abiding bot and a misc...   
2  Imagine two bots, a law-abiding bot and a misc...   
3  Imagine two bots, a law-abiding bot and a misc...   
4  Imagine two bots, a law-abiding bot and a mi

## Filter Ambiguous Intents

Remove intents that are simply marked as "ambiguous" or similar to avoid outliers in clustering.

In [3]:
# Filter out ambiguous intents
print(f"Original dataset size: {len(df)}")

# Create filter for ambiguous intents (case-insensitive)
ambiguous_pattern = r'^\s*ambiguous\s*$|^\s*uncertain\s*$|^\s*unclear\s*$|^\s*$'
df_filtered = df[~df['Intent'].str.strip().str.lower().str.match(ambiguous_pattern, na=False)].copy()

# Also remove completely empty intents
df_filtered = df_filtered[df_filtered['Intent'].notna() & (df_filtered['Intent'].str.strip() != '')].copy()

ambiguous_count = len(df) - len(df_filtered)
print(f"Filtered out {ambiguous_count} ambiguous/empty intents ({ambiguous_count/len(df)*100:.2f}%)")
print(f"Remaining dataset size: {len(df_filtered)}")

# Show some examples of filtered intents
if ambiguous_count > 0:
    filtered_intents = df[~df['Intent'].isin(df_filtered['Intent'])]['Intent'].value_counts().head(5)
    print(f"\nMost common filtered intents:")
    for intent, count in filtered_intents.items():
        print(f"  '{intent}': {count} occurrences")

Original dataset size: 1730
Filtered out 43 ambiguous/empty intents (2.49%)
Remaining dataset size: 1687

Most common filtered intents:
  'Ambiguous': 19 occurrences
  'ambiguous': 18 occurrences
  'Ambiguous ': 3 occurrences
  'ambiguous ': 3 occurrences


## Topic Modeling with BERTopic

Use BERTopic's full pipeline for automatic topic discovery with cached embeddings.

In [5]:
from intention_jailbreak.embeddings import BERTopicModelWrapper
from umap import UMAP
from hdbscan import HDBSCAN

# Configure UMAP
umap_model = UMAP(
    n_neighbors=15,      # Neighborhood size
    n_components=5,      # Dimensionality of reduced space
    min_dist=0.0,        # How tightly to pack points
    metric='cosine',
    random_state=42
)

# Configure HDBSCAN with higher minimum cluster size for fewer topics
hdbscan_model = HDBSCAN(
    min_cluster_size=15,  # Increased to get fewer, more meaningful topics
    min_samples=5,        # Minimum samples in neighborhood
    metric='euclidean',
    cluster_selection_method='eom',  # Excess of Mass
    prediction_data=True
)

# Initialize BERTopic wrapper with stronger Qwen embedding model
topic_wrapper = BERTopicModelWrapper(
    cache_dir="../data/embeddings",
    embedding_model="Qwen/Qwen3-Embedding-0.6B",  # Stronger 600M parameter model
    # BERTopic parameters
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto",    # Auto-reduce similar topics
    verbose=True
)

# Extract intent texts from FILTERED dataset
intents = df_filtered["Intent"].tolist()

print(f"Total intents to analyze (after filtering): {len(intents)}")
print(f"\nSample intents:")
for i, intent in enumerate(intents[:3], 1):
    print(f"{i}. {intent[:100]}..." if len(intent) > 100 else f"{i}. {intent}")

# Fit model and discover topics (embeddings will be cached)
topics, probs = topic_wrapper.fit_transform(
    intents, 
    use_cache=True, 
    cache_prefix="intent_qwen_bertopic_filtered"
)

# Add topics to FILTERED dataframe
df_filtered["topic"] = topics
df_filtered["topic_prob"] = probs.max(axis=1) if len(probs.shape) > 1 else probs

# Display topic information
print("\n" + "="*60)
print("DISCOVERED TOPICS (Filtered Data)")
print("="*60)
topic_info = topic_wrapper.get_topic_info()
print(topic_info)

# Show topic distribution
topics_array = np.array(topics)
print(f"\nTopic Distribution:")
print(f"  Outliers (Topic -1): {(topics_array == -1).sum()}")
print(f"  Clustered topics: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"  Average intents per topic: {len(topics_array[topics_array != -1]) / max(1, len(set(topics)) - (1 if -1 in topics else 0)):.1f}")

Total intents to analyze (after filtering): 1687

Sample intents:
1. The intent is to obtain the contents of a copyrighted comic strip
2. To obtain the contents of a copyrighted comic strip
3. obtain the contents of copyrighted comic strip
🔄 Generating embeddings with sentence-transformers...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

💾 Cached embeddings to intent_qwen_bertopic_filtered_972d55e362b6117d6903a2aef351a65a.pkl
🔄 Fitting BERTopic model...


2025-11-05 06:02:10,780 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
/home/jazhyc/projects/Intention Analysis/intention-jailbreak/.venv/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning:

The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.

/home/jazhyc/projects/Intention Analysis/intention-jailbreak/.venv/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning:

The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.

2025-11-05 06:02:21,118 - BERTopic - Dimensionality - Completed ✓
2025-11-05 06:02:21,119 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-05 06:02:21,118 - BERTopic - Dimensionality - Completed ✓
2025-11-05 06:02:21,119 - BERTop

✓ Topic modeling complete!
  Found 19 topics

DISCOVERED TOPICS (Filtered Data)
    Topic  Count                                      Name  \
0      -1    388                          -1_to_the_for_is   
1       0    346                   0_information_to_of_the   
2       1    190             1_sexual_story_write_explicit   
3       2    185           2_stereotypes_people_the_racial   
4       3    117              3_story_fictional_create_the   
5       4     51                    4_illegal_drugs_how_to   
6       5     45                  5_to_convince_how_advice   
7       6     43             6_women_gender_against_hiring   
8       7     38       7_medical_misinformation_spread_and   
9       8     32           8_fire_house_start_instructions   
10      9     31          9_ask_question_immigration_legal   
11     10     29         10_media_social_account_celebrity   
12     11     28              11_dog_animal_animals_breeds   
13     12     28      12_copyrighted_strip_comic_rep

## Explore Top Topics

Examine the top topics by size and their representative intents.

In [6]:
# Show top 10 topics (excluding outliers)
top_topics = topic_info[topic_info['Topic'] != -1].head(10)

print("TOP 10 TOPICS BY SIZE")
print("="*100)

for idx, row in top_topics.iterrows():
    topic_id = row['Topic']
    topic_name = row['Name']
    count = row['Count']
    
    print(f"\nTopic {topic_id}: {topic_name}")
    print(f"Count: {count} intents")
    print(f"Top words: {', '.join(row['Representation'][:10])}")
    
    # Show 3 representative intents from FILTERED dataframe
    topic_intents = df_filtered[df_filtered['topic'] == topic_id]['Intent'].head(3).tolist()
    print("Representative intents:")
    for i, intent in enumerate(topic_intents, 1):
        print(f"  {i}. {intent[:100]}..." if len(intent) > 100 else f"  {i}. {intent}")
    print("-"*100)

TOP 10 TOPICS BY SIZE

Topic 0: 0_information_to_of_the
Count: 346 intents
Top words: information, to, of, the, get, how, obtain, is, find, intent
Representative intents:
  1. get casting profiles for characters in a workplace sitcom
  2. To learn what happens when one is caught
  3. ask what happens if you're caught
----------------------------------------------------------------------------------------------------

Topic 1: 1_sexual_story_write_explicit
Count: 190 intents
Top words: sexual, story, write, explicit, involving, content, about, create, and, generate
Representative intents:
  1. The intent is to make a statement about a person firing their unspecified, potentially sexual load t...
  2. To write a story with potentially sexual or inappropriate double meanings involving the phrase “firi...
  3. Write a sexually explicit story about male ejaculation
----------------------------------------------------------------------------------------------------

Topic 2: 2_stereotypes_pe

In [7]:
import plotly.express as px
import plotly.graph_objects as go
from umap import UMAP

# Get embeddings for visualization
embeddings = topic_wrapper.get_embeddings()

# Reduce to 2D for visualization
umap_2d = UMAP(n_components=2, random_state=42, metric='cosine')
embeddings_2d = umap_2d.fit_transform(embeddings)

# Create dataframe for plotting
plot_df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'topic': topics_array,
    'intent': [intent[:80] + '...' if len(intent) > 80 else intent for intent in intents],
    'topic_name': [topic_info.loc[topic_info['Topic'] == t, 'Name'].values[0] 
                   if t in topic_info['Topic'].values else 'Outlier' 
                   for t in topics_array]
})

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x='x',
    y='y',
    color='topic',
    hover_data=['intent', 'topic_name'],
    title='Intent Clustering Visualization (2D UMAP Projection)',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'topic': 'Topic ID'},
    color_continuous_scale='viridis',
    width=1000,
    height=700
)

fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.update_layout(
    font=dict(size=12),
    hoverlabel=dict(font_size=10)
)

fig.show()

print(f"Visualization shows {len(set(topics_array))} distinct clusters")

Visualization shows 20 distinct clusters


## Manual Topic Labeling Interface

Interactively review topics and provide custom labels based on sample intents.

In [11]:
import json
from pathlib import Path
import random

# Initialize topic labels dictionary
topic_labels = {}

# Load existing labels if they exist
labels_file = Path('../data/annotations/topic_labels.json')
if labels_file.exists():
    with open(labels_file, 'r') as f:
        topic_labels = json.load(f)
    print(f"✓ Loaded existing topic labels from {labels_file}")
    print(f"  Found labels for {len(topic_labels)} topics")
else:
    print("No existing topic labels found. Starting fresh.")

# Get all topics excluding outliers
topics_to_label = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)

print(f"\nTopics to label: {len(topics_to_label)}")
print(f"Already labeled: {len([t for t in topic_labels.keys() if t != '-1'])}")
print(f"\nReady to start interactive labeling!")

No existing topic labels found. Starting fresh.

Topics to label: 19
Already labeled: 0

Ready to start interactive labeling!


In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Configuration
NUM_SAMPLE_PROMPTS = 10

# Create output widget for displaying topic info
output = widgets.Output()
label_input = widgets.Text(
    placeholder='Enter topic label (e.g., "Drug Manufacturing Instructions")',
    description='Topic Label:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)
submit_button = widgets.Button(description='Save & Next', button_style='success')
skip_button = widgets.Button(description='Skip', button_style='warning')
progress_label = widgets.HTML()

# State management
current_topic_idx = [0]  # Use list to make it mutable in nested function

def get_topics_to_process():
    """Get list of topics that need labeling"""
    all_topics = topics_to_label['Topic'].tolist()
    # Filter out already labeled topics
    return [t for t in all_topics if str(t) not in topic_labels]

def display_topic(topic_id):
    """Display topic information and sample intents"""
    with output:
        clear_output(wait=True)
        
        # Get topic info
        topic_row = topic_info[topic_info['Topic'] == topic_id].iloc[0]
        topic_name = topic_row['Name']
        count = topic_row['Count']
        top_words = topic_row['Representation'][:10]
        
        # Get sample intents (up to NUM_SAMPLE_PROMPTS random samples)
        topic_intents = df_filtered[df_filtered['topic'] == topic_id]['Intent'].tolist()
        sample_intents = random.sample(topic_intents, min(NUM_SAMPLE_PROMPTS, len(topic_intents)))
        
        # Display
        print("="*100)
        print(f"TOPIC {topic_id}")
        print("="*100)
        print(f"Auto-generated name: {topic_name}")
        print(f"Number of intents: {count}")
        print(f"Top keywords: {', '.join(top_words)}")
        print(f"\n{'-'*100}")
        print(f"SAMPLE INTENTS ({len(sample_intents)}):")
        print(f"{'-'*100}")
        
        for i, intent in enumerate(sample_intents, 1):
            print(f"\n{i}. {intent}")
        
        print(f"\n{'='*100}")
        
        # Check if already labeled
        if str(topic_id) in topic_labels:
            print(f"\n✓ Current label: '{topic_labels[str(topic_id)]}'")
            print("  (You can provide a new label to update it)")
        
        # Clear input field
        label_input.value = ''
        
        # Update progress
        topics_remaining = get_topics_to_process()
        progress = len(topics_to_label) - len(topics_remaining)
        progress_label.value = f"<b>Progress: {progress}/{len(topics_to_label)} topics labeled</b>"

def on_submit_clicked(button):
    """Handle submit button click"""
    topics_remaining = get_topics_to_process()
    
    if not topics_remaining:
        with output:
            clear_output(wait=True)
            print("\n✅ All topics have been labeled!")
        return
    
    topic_id = topics_remaining[0]
    label = label_input.value.strip()
    
    if not label:
        with output:
            clear_output(wait=True)
            print("⚠️ Please enter a label before saving")
            display_topic(topic_id)
        return
    
    # Save label
    topic_labels[str(topic_id)] = label
    
    # Save to file
    labels_file = Path('../data/annotations/topic_labels.json')
    labels_file.parent.mkdir(parents=True, exist_ok=True)
    with open(labels_file, 'w') as f:
        json.dump(topic_labels, f, indent=2)
    
    # Move to next topic
    topics_remaining = get_topics_to_process()
    if topics_remaining:
        display_topic(topics_remaining[0])
    else:
        with output:
            clear_output(wait=True)
            print("\n✅ All topics have been labeled!")
            print(f"Labels saved to {labels_file}")

def on_skip_clicked(button):
    """Handle skip button click"""
    topics_remaining = get_topics_to_process()
    
    if not topics_remaining:
        with output:
            clear_output(wait=True)
            print("\n✅ All topics have been labeled!")
        return
    
    # Move to next topic (current one will be skipped)
    if len(topics_remaining) > 1:
        display_topic(topics_remaining[1])
    else:
        with output:
            clear_output(wait=True)
            print("\n✅ All topics have been labeled!")

# Bind button callbacks
submit_button.on_click(on_submit_clicked)
skip_button.on_click(on_skip_clicked)

# Display initial topic
topics_remaining = get_topics_to_process()
if topics_remaining:
    print("Initializing interactive labeling interface...\n")
    
    # Display progress
    progress = len(topics_to_label) - len(topics_remaining)
    progress_label.value = f"<b>Progress: {progress}/{len(topics_to_label)} topics labeled</b>"
    
    # Display controls first so they appear on top
    controls = widgets.VBox([
        label_input,
        widgets.HBox([submit_button, skip_button]),
        progress_label
    ])
    display(controls)
    
    # Then display the topic in the output widget
    display(output)
    display_topic(topics_remaining[0])
else:
    print("✅ All topics have been labeled!")

Initializing interactive labeling interface...



Output()